# This is the code to generate the essential points in the .geo file of Nicoya
## Like before, we only define points, and leave the lines, surfaces, volume, physical entities to the GUI of *Gmsh*, easier for visualization and fine optimization. 

* After the geometry .geo file is finalized, activate the conda environment that contains the *Fenics* and *Gmsh* packages. Then use *Gmsh* to build 3D finite element mesh. Finally, use *Fenics* to convert the mesh file to regions and facets.

    ``gmsh -3 NAME.geo -format msh2 -optimize_netgen -smooth 3``
    
    ``dolfin-convert NAME.msh NAME.xml ``

In [ ]:
# import gmsh
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import utils as ut


## First write in the boundary points of cubiod domain

In [ ]:
# Define the data directory
datadir = "./"

# Define the geo file name
output_dir = "../mesh/"
output_geo_file = "test_nicoya.geo"

# Define sep
sep = "//+"

# Define the cubiod boundary points, in meters
xmin, xmax = -1000e3, 1000e3
ymin, ymax = -1000e3, 1000e3
zmin, zmax = -200e3, 0.0
print(xmin, xmax, ymin, ymax, zmin, zmax)

# write to the .geo file 
geo_content = f"""\
{sep}
xmin = {xmin};
xmax = {xmax};
ymin = {ymin};
ymax = {ymax};
zmin = {zmin};
zmax = {zmax}; 

"""

# Write to the file
with open(output_dir + output_geo_file, "w") as f:
    f.write(geo_content)


# define the anomaly??

# Define the mesh sizes for different regions, in meters
dx = 200e3
dx_fault = 4e3
dx_fault_coarse = 20e3
dx_anomaly = 4e3
dx_anomaly_coarse = 20e3

geo_content = f"""\
{sep}
dx = {dx};
dx_fault = {dx_fault};
dx_fault_coarse = {dx_fault_coarse};
dx_anomaly = {dx_anomaly};
dx_anomaly_coarse = {dx_anomaly_coarse};

"""

# append to the file
with open(output_dir + output_geo_file, "a") as f:
    f.write(geo_content)

In [ ]:
# Define the 8 corner points
ncorners = 8
corner_points = [
    'xmin, ymax, zmax, dx', 
    'xmax, ymax, zmax, dx', 
    'xmax, ymin, zmax, dx', 
    'xmin, ymin, zmax, dx', 
    'xmin, ymax, zmin, dx',
    'xmax, ymax, zmin, dx',
    'xmax, ymin, zmin, dx',
    'xmin, ymin, zmin, dx',
]

# Append boundary points to the geo file
start_index = 1
with open(output_dir + output_geo_file, "a") as f:
    for i, coords in enumerate(corner_points, start=start_index):
        f.write(f"{sep}\nPoint({i}) = {{{coords}}};\n")
start_index += ncorners 

## Next write in the trench points

In [ ]:
# Define the trench file
trench_file = "trench.gmt.inuse"

# Filter by both longitude and latitude
segments = ut.parse_trench_file(datadir + trench_file, 
                           lon_range=(272-360, 276-360), 
                           lat_range=(8, 11.5))

# Convert to DataFrame for analysis
trench = ut.segments_to_dataframe(segments)

# define the centroid of relative coordinates
lon0, lat0 = -85.5+360, 10

# convert to relative locations in km
trench['x'], trench['y'] = ut.azi_equidist_proj(trench['lon'], trench['lat'], lon0, lat0)

# Convert coordinates from km to meters and round to the nearest meter
trench['x'] = (trench['x'] * 1000).round()
trench['y'] = (trench['y'] * 1000).round()
trench['z'] = 0.0

print(trench.head())

In [ ]:
# Add a top boundary point at ymax
t_bound = pd.DataFrame([{'x': trench['x'].min().astype(np.float64), 'y': ymax, 'z': 0}])
trench = pd.concat([t_bound, trench], axis=0, ignore_index=True)

# Add a right boundary point at xmax
r_bound = pd.DataFrame([{'x': xmax, 'y': trench['y'].min().astype(np.float64), 'z': 0}])
trench = pd.concat([trench, r_bound], axis=0, ignore_index=True)

# Sort the extracted data by x-values for consistency
trench = trench.sort_values(by='x').reset_index(drop=True)    

# 2D list to store indices for each depth z, including z=0
point_indices_per_depth = []
        
# for the surface, ie., z=0, use the coarse trench data
trench_points = []
point_indices = []  # Store indices for the current depth

# Format and append the first point with fine resolution
tmp = trench.iloc[0]
point_index = start_index  # Compute the actual index
point_indices.append(point_index)  # Store the index for this depth
trench_points.append(f"{sep}\nPoint({point_index}) = {{{tmp.x:.6f}, {tmp.y:.6f}, {tmp.z:.6f}, dx}};")

# Loop over the middle points and format them with standard fault resolution
for i, row in enumerate(trench.iloc[1:-1].itertuples(index=False), start=1):
    point_index = start_index+i
    point_indices.append(point_index)
    trench_points.append(f"{sep}\nPoint({point_index}) = {{{row.x:.6f}, {row.y:.6f}, {row.z:.6f}, dx_fault_coarse}};")

# Format and append the last point with fine resolution
tmp = trench.iloc[-1]
point_index = start_index+trench.iloc[:-1].shape[0]
point_indices.append(point_index)
trench_points.append(f"{sep}\nPoint({point_index}) = {{{tmp.x:.6f}, {tmp.y:.6f}, {tmp.z:.6f}, dx}};")

# # Loop over with standard dx resolution
# for i, row in enumerate(trench.itertuples(index=False), start=1):
#     point_index = start_index + i - 1  # Compute the actual index
#     point_indices.append(point_index)  # Store the index for this depth
#     trench_points.append(f"{sep}\nPoint({point_index}) = {{{row.x:.6f}, {row.y:.6f}, {row.z:.6f}, dx}};")

point_indices_per_depth.append(point_indices)  # Store the indices list for this depth

# Append formatted points to the output .geo file
with open(output_dir + output_geo_file, "a") as f:
    f.write("\n".join(trench_points) + "\n")

# Increment the start index for the next set of points
start_index += trench.shape[0]
print(start_index)

## Next deal with slab interface

In [ ]:
# Define the plate interface file
plate_file = "nicoya_slab2_100_10_10.txt"

plate = ut.parse_plate_interface_file(datadir + plate_file)

plate.head()

plate.tail()


In [ ]:
# convert to relative locations in km
plate['x'], plate['y'] = ut.azi_equidist_proj(plate['lon'], plate['lat'], lon0, lat0)

# Convert coordinates from km to meters and round to the nearest meter
plate['x'] = (plate['x'] * 1000).round()
plate['y'] = (plate['y'] * 1000).round()
plate['z'] = (plate['dep'] * 1000).round()

print(plate['lon'].min(), plate['lon'].max())
print(plate['lat'].min(), plate['lat'].max())

print(plate['x'].min(), plate['x'].max())
print(plate['y'].min(), plate['y'].max())


In [ ]:
# define the boundary of the fault region
lonA, latA, lonB, latB = -87+360, 10, -85.5+360, 11.5
lonC, latC, lonD, latD = -85.4+360, 8.8, -85.4+360+1.5, 8.8+1.5

# Convert coordinates from degrees to relative locations in km
xA, yA = ut.azi_equidist_proj(lonA, latA, lon0, lat0)
xB, yB = ut.azi_equidist_proj(lonB, latB, lon0, lat0)
xC, yC = ut.azi_equidist_proj(lonC, latC, lon0, lat0)
xD, yD = ut.azi_equidist_proj(lonD, latD, lon0, lat0)

# Calculate slope and intercept of a line
def line_m_q(p1, p2):
    m = ( p2[1] - p1[1] ) / ( p2[0] - p1[0] )   # slope
    q = ( p1[0]*p2[1] - p2[0]*p1[1] ) / ( p1[0] - p2[0] )  # intercept
    return m, q

m_AB, q_AB = line_m_q(np.array([xA, yA]), np.array([xB, yB]))
print(f"m_lt: {m_AB}, q_lt: {q_AB}")

m_CD, q_CD = line_m_q(np.array([xC, yC]), np.array([xD, yD]))
print(f"m_lt: {m_CD}, q_lt: {q_CD}")

In [ ]:
z_target = np.array(plate['z'].unique())

fig, ax = plt.subplots(1, 1, figsize=(6, 6))

for i in range(len(z_target)):
    # i =0
    # Get the current depth value
    dep = z_target[i]
    # print(dep)
    # Extract plate data at the exact depth value from 'plate_cut'
    plate_dum = plate.loc[plate['z'] == dep]
    plate_dum = plate_dum.sort_values(by='x').reset_index(drop=True) 
    # print(plate_dum)

    ax.plot(plate_dum['x']/1e3, plate_dum['y']/1e3, linestyle='-', color='gray', linewidth=1)

    y_line = m_AB * plate_dum['x']/1e3 + q_AB

    # Compute vertical distance between line and curve
    distance = np.abs(plate_dum['y']/1e3 - y_line)

    # Find index of minimal distance
    idx_min = np.argmin(distance)
    x_int = plate_dum['x'][idx_min]/1e3
    y_int = plate_dum['y'][idx_min]/1e3  # or m * x_int + b for the line's value

    print(f"Depth: {dep/1e3}, x_int: {x_int}, y_int: {y_int}")

    plt.plot(x_int, y_int, 'bo', label='Intersection (approx)')


    y_line = m_CD * plate_dum['x']/1e3 + q_CD

    # Compute vertical distance between line and curve
    distance = np.abs(plate_dum['y']/1e3 - y_line)

    # Find index of minimal distance
    idx_min = np.argmin(distance)
    x_int = plate_dum['x'][idx_min]/1e3
    y_int = plate_dum['y'][idx_min]/1e3  # or m * x_int + b for the line's value

    print(f"Depth: {dep/1e3}, x_int: {x_int}, y_int: {y_int}")

    plt.plot(x_int, y_int, 'bo', label='Intersection (approx)')


ax.plot(trench['x']/1e3, trench['y']/1e3, linestyle='-', color='black', linewidth=1.5)

ax.set_ylim(-250, 250)
ax.set_xlim(-300, 250)


In [ ]:
z_target = np.array(plate['z'].unique())
print(z_target)

# Initialize an empty DataFrame to store all plate depth values
# It uses the same column names as the 'plate' DataFrame.
plate_dep_all = pd.DataFrame(columns=plate.columns)

# Loop over each depth level in z_target
for i in range(len(z_target)):
# i = 0

    # Get the current depth value
    dep = z_target[i]

    # Extract plate data at the exact depth value from 'plate_cut'
    plate_dum = plate.loc[plate['z'] == dep]
    
    # Sort the extracted data by x-values for consistency
    plate_dum = plate_dum.sort_values(by='x').reset_index(drop=True) 
    
    # make it coarser, use every 10th point
    plate_dep = plate_dum.iloc[::2].copy()
    print(plate_dep.shape)

    # Sort the extracted data by x-values for consistency
    plate_dep = plate_dep.sort_values(by='x').reset_index(drop=True) 
    # print(plate_dep.head())   

    # Add a top boundary point at ymax
    t_bound = pd.DataFrame([{'x': plate_dep['x'].min(), 'y': ymax, 'z': dep}])
    plate_dep = pd.concat([t_bound, plate_dep], axis=0, ignore_index=True)

    # Add a right boundary point at xmax
    r_bound = pd.DataFrame([{'x': xmax, 'y': plate_dep['y'].min(), 'z': dep}])
    plate_dep = pd.concat([plate_dep, r_bound], axis=0, ignore_index=True)

    # Append the processed plate depth data to the combined DataFrame
    plate_dep_all = pd.concat([plate_dep_all, plate_dep], axis=0, ignore_index=True)

    # Prepare a list to store formatted points for output
    plate_dep_points = []
    point_indices = []  # Store indices for the current depth

    # Format and append the first point with fine resolution
    tmp = plate_dep.iloc[0]
    point_index = start_index  # Compute the actual index
    point_indices.append(point_index)  # Store the index for this depth
    plate_dep_points.append(f"{sep}\nPoint({point_index}) = {{{tmp.x:.6f}, {tmp.y:.6f}, {tmp.z:.6f}, dx}};")

    # Loop over the middle points and format them with standard fault resolution
    for i, row in enumerate(plate_dep.iloc[1:-1].itertuples(index=False), start=1):
        point_index = start_index+i
        point_indices.append(point_index)
        plate_dep_points.append(f"{sep}\nPoint({point_index}) = {{{row.x:.6f}, {row.y:.6f}, {row.z:.6f}, dx_fault_coarse}};")

    # Format and append the last point with fine resolution
    tmp = plate_dep.iloc[-1]
    point_index = start_index+plate_dep.iloc[:-1].shape[0]
    point_indices.append(point_index)
    plate_dep_points.append(f"{sep}\nPoint({point_index}) = {{{tmp.x:.6f}, {tmp.y:.6f}, {tmp.z:.6f}, dx}};")

    # Append formatted points to the output .geo file
    with open(output_dir + output_geo_file, "a") as f:
        f.write("\n".join(plate_dep_points) + "\n")
        
    # Increment the start index for the next set of points
    start_index += plate_dep.shape[0]
    print(start_index)

    point_indices_per_depth.append(point_indices)  # Store the indices list for this depth

## Finally, deal with the bottom layer

In [ ]:
# for the z=zmin boundary, we just use the same points as the last depth level of plate model with z=zmin
plate_bottom = plate_dep.copy()
plate_bottom['z'] = zmin

plate_bottom_points = []
point_indices = []  # Store indices for the current depth
# Loop over with standard dx resolution
for i, row in enumerate(plate_bottom.itertuples(index=False), start=1):
    point_index = start_index + i - 1  # Compute the actual index
    point_indices.append(point_index)  # Store the index for this depth
    plate_bottom_points.append(f"{sep}\nPoint({point_index}) = {{{row.x:.6f}, {row.y:.6f}, {row.z:.6f}, dx}};")

# Append formatted points to the output .geo file
with open(output_dir + output_geo_file, "a") as f:
    f.write("\n".join(plate_bottom_points) + "\n")

# Increment the start index for the next set of points
start_index += plate_bottom.shape[0]
print(start_index)

point_indices_per_depth.append(point_indices)  # Store the indices list for this depth